In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [14]:
df = pd.read_csv('train.csv')
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
404,405,0,3,"Oreskovic, Miss. Marija",female,20.0,0,0,315096,8.6625,NaN,S
578,579,0,3,"Caram, Mrs. Joseph (Maria Elias)",female,NaN,1,0,2689,14.4583,NaN,C
195,196,1,1,"Lurette, Miss. Elise",female,58.0,0,0,PC 17569,146.5208,B80,C
612,613,1,3,"Murphy, Miss. Margaret Jane",female,NaN,1,0,367230,15.5000,NaN,Q
429,430,1,3,"Pickard, Mr. Berk (Berk Trembisky)",male,32.0,0,0,SOTON/O.Q. 392078,8.0500,E10,S


In [15]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace=True)
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
601,0,3,male,NaN,0,0,7.8958,S
763,1,1,female,36.0,1,2,120.0000,S
267,1,3,male,25.0,1,0,7.7750,S
805,0,3,male,31.0,0,0,7.7750,S
588,0,3,male,22.0,0,0,8.0500,S


In [16]:
# Step 1 : train test split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), 
                                                    df['Survived'], 
                                                    test_size=0.2, 
                                                    random_state=42)

In [17]:
X_train.sample(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
230,1,female,35.0,1,0,83.475,S
274,3,female,NaN,0,0,7.750,Q
47,3,female,NaN,0,0,7.750,Q
766,1,male,NaN,0,0,39.600,C
748,1,male,19.0,1,0,53.100,S


In [18]:
# check if there is missing values present if yes fill them using SimpleImputer object
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [19]:
# Applying Imputation -> filling missing values
si_age = SimpleImputer()    # default strategy -> mean
si_embarked = SimpleImputer(strategy='most_frequent')    # to fill missing with most frequent values

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

# missing values filled with applied strategy

In [20]:
# check the categorical features and encode them with OneHotEncoder
df.sample()

# here output is already labeled(1, 0) not need of LabelEncoder
# Sex and Embarked can be encoded

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
381,1,3,female,1.0,0,2,15.7417,C


In [24]:
# one hot encoding Sex and Embarked
ohe_sex = OneHotEncoder(sparse=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse=False, handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train[['Embarked']])

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test[['Embarked']])

/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [27]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

In [28]:
X_train_transformed = np.concatenate((X_train_rem, X_train_age, X_train_sex, X_train_embarked), axis=1)
X_test_transformed = np.concatenate((X_test_rem, X_test_age, X_test_sex, X_test_embarked), axis=1)

In [47]:
 X_train_transformed.shape , X_test_transformed.shape

((712, 11), (179, 11))

In [34]:
# fit the model on training data
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed, y_train)

DecisionTreeClassifier()

In [35]:
# predict the values over test data
y_pred = clf.predict(X_test_transformed)

In [38]:
# y_pred

In [42]:
# Measure Accuracy Score
from sklearn.metrics import accuracy_score
accuracy_score(y_pred, y_test)


0.770949720670391

In [43]:
import pickle

In [45]:
pickle.dump(ohe_sex, open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf, open('models/clf.pkl','wb'))